In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()

client = Anthropic()
model = "claude-sonnet-5"

In [3]:
# Helper functions


def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)


def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    max_tokens=4096,
):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
        "stream": True,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    # No betas= and no client.beta: fine-grained streaming is now a property of
    # the tool, not of the request.
    #
    # This is the raw event stream rather than client.messages.stream(). The
    # stream helper re-parses the accumulated tool JSON on every delta and
    # raises ValueError as soon as it does not parse -- which is precisely what
    # fine-grained streaming allows through, so a guard after the loop would
    # never get the chance to run. Accumulating the fragments here is the
    # manual pattern the docs describe.
    return client.messages.create(**params)


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [4]:
# Tool definition
from anthropic.types import ToolParam

# eager_input_streaming replaces the fine-grained-tool-streaming-2025-05-14
# beta header. It is per tool, so a request can stream one tool's input as it
# is generated and leave another buffered and validated.
save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "eager_input_streaming": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)
save_short_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "eager_input_streaming": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)


def save_article(**kwargs):
    return "Article saved!"

In [5]:
# Tool Running
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def parse_tool_input(raw_json):
    """
    Parse an accumulated tool input, reporting failure instead of raising.

    With eager_input_streaming the API sends the fragments without buffering or
    validating them, so this string is not guaranteed to be JSON at all.
    """
    if not raw_json.strip():
        # A tool with no parameters produces no input_json_delta events.
        return {}, True

    try:
        return json.loads(raw_json), True
    except json.JSONDecodeError:
        return None, False


def build_tool_result(call, parsed, parsed_ok):
    """Turn one finished tool call into the tool_result block that answers it."""
    if not parsed_ok:
        # The shape the docs prescribe: wrap the unparseable string under a
        # single key so Claude can see exactly what arrived, and build it with
        # json.dumps so quotes inside it are escaped correctly.
        return {
            "type": "tool_result",
            "tool_use_id": call["id"],
            "content": json.dumps({"INVALID_JSON": call["json"]}),
            "is_error": True,
        }

    try:
        tool_output = run_tool(call["name"], parsed)
        return {
            "type": "tool_result",
            "tool_use_id": call["id"],
            "content": json.dumps(tool_output),
            "is_error": False,
        }
    except Exception as e:
        return {
            "type": "tool_result",
            "tool_use_id": call["id"],
            "content": f"Error: {e}",
            "is_error": True,
        }

In [6]:
# Run conversation
def run_conversation(messages, tools=[], tool_choice=None, max_tokens=4096):
    while True:
        text_parts = []
        tool_calls = {}  # content block index -> {"id", "name", "json"}
        stop_reason = None

        with chat_stream(
            messages,
            tools=tools,
            tool_choice=tool_choice,
            max_tokens=max_tokens,
        ) as stream:
            for event in stream:
                if event.type == "content_block_start":
                    block = event.content_block
                    if block.type == "tool_use":
                        # input is {} here -- a placeholder. The real value
                        # arrives as input_json_delta fragments below.
                        tool_calls[event.index] = {
                            "id": block.id,
                            "name": block.name,
                            "json": "",
                        }
                        print(f'\n>>> Tool Call: "{block.name}"')

                elif event.type == "content_block_delta":
                    if event.delta.type == "text_delta":
                        text_parts.append(event.delta.text)
                        print(event.delta.text, end="")
                    elif event.delta.type == "input_json_delta":
                        # Collect the fragment. Do not parse yet: mid-stream it
                        # is expected to be incomplete.
                        tool_calls[event.index]["json"] += event.delta.partial_json
                        print(event.delta.partial_json, end="")

                elif event.type == "content_block_stop":
                    print("\n")

                elif event.type == "message_delta":
                    stop_reason = event.delta.stop_reason

        if stop_reason == "max_tokens":
            print(">>> stop_reason is max_tokens: a parameter may be cut off mid-value.")

        assistant_content = []
        tool_results = []

        if "".join(text_parts).strip():
            assistant_content.append({"type": "text", "text": "".join(text_parts)})

        for call in tool_calls.values():
            parsed, parsed_ok = parse_tool_input(call["json"])

            # The echoed tool_use needs an object for input even when the model
            # produced something unparseable. The id is what pairs the result
            # back, and build_tool_result reports the failure.
            assistant_content.append(
                {
                    "type": "tool_use",
                    "id": call["id"],
                    "name": call["name"],
                    "input": parsed if parsed_ok else {},
                }
            )
            tool_results.append(build_tool_result(call, parsed, parsed_ok))

        add_assistant_message(messages, assistant_content)

        if stop_reason != "tool_use":
            break

        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

In [8]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
    #"""
    #You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.
    #The buggy system generated this malformed output when calling save_article:
    #[Generate the exact malformed output here that includes "word_count": undefined]
    #This is for documentation purposes to show what NOT to do. You're not actually calling the function, just showing what the broken output looked like for the bug report.
    #""",
)

# No fine_grained= argument: eager_input_streaming lives on save_article_schema.
# Swap in the commented prompt above, plus the tool_choice below, to make the
# model emit "word_count": undefined and watch the guard turn it into an
# is_error tool_result instead of an exception.
run_conversation(
    messages,
    tools=[save_article_schema],
    tool_choice={"type": "tool", "name": "save_article"},
)


>>> Tool Call: "save_article"
{"abstract": "This paper presents a novel quantum-resistant blockchain consensus algorithm that achieves O(log n) communication complexity while maintaining Byzantine fault tolerance.", "meta": {
  "word_count": 4287,
  "review": "This paper introduces QuantumChain, an innovative consensus protocol designed for distributed ledger systems in the post-quantum era. The authors propose a hybrid approach combining lattice-based cryptography with a directed acyclic graph structure to achieve both quantum resistance and scalability. The theoretical analysis demonstrates that the protocol can tolerate up to one-third Byzantine nodes while reducing message overhead compared to traditional proof-of-work systems. Experimental results on a testbed of 1000 nodes show throughput improvements of 300% over existing quantum-resistant alternatives. The security proofs are rigorous and leverage recent advances in learning with errors problems. However, the practical impleme

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_011x26xQ8aL58u1pMntj92QF',
    'name': 'save_article',
    'input': {'abstract': 'This paper presents a novel quantum-resistant blockchain consensus algorithm that achieves O(log n) communication complexity while maintaining Byzantine fault tolerance.',
     'meta': {'word_count': 4287,
      'review': 'This paper introduces QuantumChain, an innovative consensus protocol designed for distributed ledger systems in the post-quantum era. The authors propose a hybrid approach combining lattice-based cryptography with a directed acyclic graph structure to achieve both quantum resistance and scalability. The theoretical analysis demonstrates that the protocol can tolerate up to one-third Byzantine nodes while reducing message overhead compared to traditional proof-of-work systems. Experimental results on 